# Comparative Analysis of CNN and Efficient Vision Transformers for Gleason Grade Classification

**Course:** CSELEC2C Machine Learning (2025–2026)  
**Models:** ResNet-50 · TinyViT-11M · FastViT-T8  
**Dataset:** SICAPv2 (18,783 patches, 4-class: NC / G3 / G4 / G5)

---
This notebook runs end-to-end: dataset check → training → evaluation → Grad-CAM → results table.

## 0. Environment Setup

In [ ]:
# Install dependencies (Colab)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'timm==1.0.3', 'PyYAML', 'openpyxl', 'matplotlib', 'pandas', 'seaborn', 'statsmodels'])

In [ ]:
# Mount Google Drive if running in Colab (optional — for checkpoint saving)
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    print('Google Drive mounted.')
except ImportError:
    IN_COLAB = False
    print('Not running in Colab — Drive not mounted.')

In [ ]:
import os, sys
from pathlib import Path

# ── Project root ─────────────────────────────────────────────────────────────
# SET THIS to where your ml_project_implementation folder lives on Drive.
# Example: '/content/drive/MyDrive/gleason_project'
PROJECT_ROOT = Path('/content/drive/MyDrive/gleason_project')

# Fallback for local Jupyter runs (cwd is the project root or notebooks/)
if not PROJECT_ROOT.exists():
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

print(f'Project root: {PROJECT_ROOT}')
assert PROJECT_ROOT.exists(), f'Project root does not exist: {PROJECT_ROOT}'
assert (PROJECT_ROOT / 'data' / 'dataset.py').exists(), \
    f'data/dataset.py not found under {PROJECT_ROOT} — check the path'

# Make project modules importable + cd into project root for relative paths
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

## 1. SICAPv2 Dataset Check

The notebook expects the dataset at `<project_root>/SICAPv2/`.  
If not found, download from https://data.mendeley.com/datasets/9xxm58dvs3/ and extract there.

In [ ]:
SICAP_ROOT = PROJECT_ROOT / 'SICAPv2'

REQUIRED_PATHS = [
    SICAP_ROOT / 'images',
    SICAP_ROOT / 'masks',
    SICAP_ROOT / 'partition' / 'Test' / 'Test.xlsx',
    SICAP_ROOT / 'partition' / 'Test' / 'Train.xlsx',
    SICAP_ROOT / 'partition' / 'Validation' / 'Val1' / 'Train.xlsx',
    SICAP_ROOT / 'partition' / 'Validation' / 'Val1' / 'Test.xlsx',
]

if not SICAP_ROOT.exists():
    raise FileNotFoundError(
        f'SICAPv2 not found at {SICAP_ROOT}.\n'
        'Download from: https://data.mendeley.com/datasets/9xxm58dvs3/\n'
        'and extract so the folder structure is:\n'
        '  <project_root>/SICAPv2/images/   <-- .jpg patches\n'
        '  <project_root>/SICAPv2/partition/'
    )

missing = [p for p in REQUIRED_PATHS if not p.exists()]
if missing:
    raise FileNotFoundError('Missing SICAPv2 files:\n' + '\n'.join(f'  {p}' for p in missing))

n_images = len(list((SICAP_ROOT / 'images').glob('*.jpg')))
print(f'SICAPv2 OK — {n_images} patches found at {SICAP_ROOT}')

## 2. Configuration

In [ ]:
import yaml

CONFIG_PATH = PROJECT_ROOT / 'training' / 'config.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Optionally enable Drive checkpointing when running in Colab
cfg['save_to_drive'] = IN_COLAB

print('Config loaded:')
for k, v in cfg.items():
    if not isinstance(v, dict):
        print(f'  {k}: {v}')

In [ ]:
# Set to False to skip training and load saved best checkpoints instead.
RUN_TRAINING = False
print(f'RUN_TRAINING={RUN_TRAINING}')

## 3. Dataset & DataLoaders

In [ ]:
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

from data.dataset import SICAPv2Dataset, CLASS_NAMES
from data.transforms import get_train_transforms, get_val_transforms

train_ds = SICAPv2Dataset(
    xlsx_path=PROJECT_ROOT / cfg['train_xlsx'],
    images_dir=PROJECT_ROOT / cfg['images_dir'],
    transform=get_train_transforms(cfg['input_size']),
)
val_ds = SICAPv2Dataset(
    xlsx_path=PROJECT_ROOT / cfg['val_xlsx'],
    images_dir=PROJECT_ROOT / cfg['images_dir'],
    transform=get_val_transforms(cfg['input_size']),
)
test_ds = SICAPv2Dataset(
    xlsx_path=PROJECT_ROOT / cfg['test_xlsx'],
    images_dir=PROJECT_ROOT / cfg['images_dir'],
    transform=get_val_transforms(cfg['input_size']),
)

class_weights = train_ds.get_class_weights()
sample_weights = [class_weights[lbl].item() for lbl in train_ds.labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=cfg['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')
print(f'Classes: {CLASS_NAMES}')
print(f'Class weights: {dict(zip(CLASS_NAMES, class_weights.tolist()))}')

In [ ]:
# Quick sanity: display one patch per class
import matplotlib.pyplot as plt
import numpy as np
from data.transforms import IMAGENET_MEAN, IMAGENET_STD

def denorm(t):
    img = t.numpy().transpose(1, 2, 0)
    return np.clip(img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)

seen = {}
for img, lbl in train_ds:
    if lbl not in seen:
        seen[lbl] = img
    if len(seen) == 4:
        break

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (lbl, img) in zip(axes, sorted(seen.items())):
    ax.imshow(denorm(img))
    ax.set_title(CLASS_NAMES[lbl])
    ax.axis('off')
plt.suptitle('One patch per class (training set)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Models

In [ ]:
from models.load_models import create_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

models = {}
for key, model_cfg in cfg['models'].items():
    m = create_model(model_cfg['timm_name'], num_classes=cfg['num_classes'])
    models[key] = m.to(device)
    params = sum(p.numel() for p in m.parameters())
    print(f'  {key}: {params/1e6:.1f}M params')

## 5. Training

Two-phase strategy:
- **Phase 1 (epochs 1–5):** freeze backbone, train head only
- **Phase 2 (epochs 6–20):** unfreeze full model, fine-tune end-to-end

In [ ]:
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast

from models.classifier_heads import freeze_backbone, unfreeze_all
from training.utils import AverageMeter, save_checkpoint, copy_checkpoint_to_drive, set_seed

set_seed(cfg['seed'])

class_weights_device = class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_device)

all_history = {}
loaded_checkpoints = {}

if RUN_TRAINING:
    for model_key, model in models.items():
        print(f'\n{"="*60}')
        print(f'Training: {model_key}')
        print(f'{"="*60}')

        freeze_backbone(model)
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'],
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])
        scaler = GradScaler() if cfg['mixed_precision'] and device.type == 'cuda' else None

        ckpt_dir = PROJECT_ROOT / cfg['checkpoint_dir'] / model_key
        best_val_acc = 0.0
        history = []

        for epoch in range(1, cfg['epochs'] + 1):
            if epoch == cfg['unfreeze_epoch']:
                print(f'  Epoch {epoch}: Unfreezing full model.')
                unfreeze_all(model)
                optimizer = torch.optim.AdamW(
                    model.parameters(), lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'],
                )
                scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=cfg['epochs'] - epoch + 1,
                )

            # --- train ---
            model.train()
            tloss, tcorrect, ttotal = AverageMeter(), 0, 0
            for imgs, lbls in train_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad()
                with autocast(enabled=scaler is not None):
                    out = model(imgs)
                    loss = criterion(out, lbls)
                if scaler:
                    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                else:
                    loss.backward(); optimizer.step()
                tloss.update(loss.item(), lbls.size(0))
                tcorrect += (out.argmax(1) == lbls).sum().item()
                ttotal += lbls.size(0)

            # --- val ---
            model.eval()
            vloss, vcorrect, vtotal = AverageMeter(), 0, 0
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    out = model(imgs)
                    loss = criterion(out, lbls)
                    vloss.update(loss.item(), lbls.size(0))
                    vcorrect += (out.argmax(1) == lbls).sum().item()
                    vtotal += lbls.size(0)

            scheduler.step()

            train_acc = tcorrect / ttotal
            val_acc   = vcorrect / vtotal
            is_best = val_acc > best_val_acc
            if is_best:
                best_val_acc = val_acc

            ckpt_name = f'epoch_{epoch:02d}.pt'
            state = dict(epoch=epoch, model_key=model_key,
                         state_dict=model.state_dict(),
                         val_acc=val_acc, optimizer=optimizer.state_dict())
            save_checkpoint(state, ckpt_dir, ckpt_name, is_best=is_best)

            if cfg.get('save_to_drive') and IN_COLAB:
                copy_checkpoint_to_drive(
                    ckpt_dir / ckpt_name,
                    Path(cfg['drive_checkpoint_dir']) / model_key,
                )

            row = dict(epoch=epoch,
                       train_loss=round(tloss.avg,4), train_acc=round(train_acc,4),
                       val_loss=round(vloss.avg,4),   val_acc=round(val_acc,4))
            history.append(row)
            marker = ' *' if is_best else ''
            print(f'  [{epoch:02d}/{cfg["epochs"]}] '
                  f'train {train_acc:.4f} | val {val_acc:.4f}{marker}')

        all_history[model_key] = history
        print(f'  Best val acc: {best_val_acc:.4f}')
else:
    for model_key, model in models.items():
        ckpt_dir = PROJECT_ROOT / cfg['checkpoint_dir'] / model_key
        best_ckpts = sorted(ckpt_dir.glob('best_*.pt'))
        if not best_ckpts:
            raise FileNotFoundError(
                f'No best checkpoint found for {model_key} in {ckpt_dir}. '
                'Set RUN_TRAINING=True to train from scratch.'
            )
        ckpt = torch.load(best_ckpts[-1], map_location=device)
        model.load_state_dict(ckpt['state_dict'])
        loaded_checkpoints[model_key] = str(best_ckpts[-1])
        print(
            f"Loaded {model_key} from {best_ckpts[-1].name} "
            f"(epoch={ckpt['epoch']}, val_acc={ckpt['val_acc']:.4f})"
        )

In [ ]:
# Plot training curves when training was run in this session
if all_history:
    fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 4), sharey=True)
    if len(models) == 1:
        axes = [axes]
    for ax, (key, hist) in zip(axes, all_history.items()):
        epochs = [r['epoch'] for r in hist]
        ax.plot(epochs, [r['train_acc'] for r in hist], label='train')
        ax.plot(epochs, [r['val_acc']   for r in hist], label='val')
        ax.axvline(cfg['freeze_epochs'], color='gray', linestyle='--', linewidth=0.8, label='unfreeze')
        ax.set_title(key)
        ax.set_xlabel('Epoch')
        ax.legend()
    axes[0].set_ylabel('Accuracy')
    plt.suptitle('Training curves', fontsize=13)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / cfg['figures_dir'] / 'training_curves.png', dpi=150)
    plt.show()
else:
    print('Training skipped in this session; no training curves to plot.')

## 6. Evaluation on Test Set

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

test_results = {}
all_preds = {}
all_labels = None

for key, model in models.items():
    model.eval()
    preds_list, labels_list = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            preds_list.append(out.argmax(dim=1).cpu())
            labels_list.append(lbls)

    all_preds[key] = torch.cat(preds_list).numpy()
    if all_labels is None:
        all_labels = torch.cat(labels_list).numpy()

    acc = float((all_preds[key] == all_labels).mean())
    num_params = sum(p.numel() for p in model.parameters())
    test_results[key] = {
        'test_accuracy': round(acc, 4),
        'num_params': num_params,
    }

    print(f"\n{'=' * 55}")
    print(f'  {key}')
    print(f"{'=' * 55}")
    print(classification_report(
        all_labels,
        all_preds[key],
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

figures_dir = PROJECT_ROOT / cfg['figures_dir']
figures_dir.mkdir(parents=True, exist_ok=True)

def plot_confusion_matrices(all_preds, all_labels, class_names, save_path=None):
    n = len(all_preds)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, (name, preds) in zip(axes, all_preds.items()):
        cm = confusion_matrix(all_labels, preds)
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        annot = np.array([
            [f"{cm[i, j]}\n({cm_norm[i, j]:.1%})" for j in range(len(class_names))]
            for i in range(len(class_names))
        ])

        sns.heatmap(
            cm_norm,
            annot=annot,
            fmt='',
            cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            vmin=0,
            vmax=1,
            ax=ax,
            linewidths=0.5,
            linecolor='gray',
            annot_kws={'size': 9},
        )
        acc = (preds == all_labels).mean()
        ax.set_title(f'{name}\nAcc: {acc:.2%}', fontweight='bold')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_confusion_matrices(
    all_preds,
    all_labels,
    CLASS_NAMES,
    save_path=figures_dir / 'confusion_matrices.png',
)

In [ ]:
from sklearn.metrics import f1_score

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(CLASS_NAMES))
width = 0.25
colors = {'resnet50': '#e15759', 'tiny_vit_11m': '#4e79a7', 'fastvit_t8': '#59a14f'}

for i, (name, preds) in enumerate(all_preds.items()):
    f1s = f1_score(all_labels, preds, average=None, zero_division=0)
    ax.bar(x + i * width, f1s, width, label=name, color=colors.get(name, 'gray'))

ax.set_xticks(x + width)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Score by Model')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / 'per_class_f1.png', dpi=150)
plt.show()

## 6.5 Statistical Comparison

Significance testing on matched test-set predictions across all three models.

In [ ]:
from scipy.stats import chi2

correct_matrix = np.column_stack([
    (all_preds[k] == all_labels).astype(int) for k in all_preds
])
model_names = list(all_preds.keys())

n, k = correct_matrix.shape
row_sums = correct_matrix.sum(axis=1)
col_sums = correct_matrix.sum(axis=0)
T = row_sums.sum()

Q_num = (k - 1) * (k * (col_sums ** 2).sum() - T ** 2)
Q_den = k * T - (row_sums ** 2).sum()
Q = Q_num / Q_den
p_cochran = 1 - chi2.cdf(Q, df=k - 1)

cochran_q_result = {
    'statistic': float(Q),
    'df': int(k - 1),
    'p_value': float(p_cochran),
}

print(f"Cochran's Q = {Q:.4f},  df = {k - 1},  p = {p_cochran:.6f}")
if p_cochran < 0.05:
    print('=> Reject H0: at least one model has a significantly different error rate.')
    print('   Proceed to pairwise McNemar tests.')
else:
    print('=> Fail to reject H0: no significant difference among the three models.')

In [ ]:
from itertools import combinations
from statsmodels.stats.contingency_tables import mcnemar

alpha_corrected = 0.05 / 3
pairwise_mcnemar_results = {}

print(f"{'Pair':<35} {'chi^2':>8} {'p-value':>10} {'Significant?':>14}")
print('-' * 70)

for i, j in combinations(range(len(model_names)), 2):
    a_correct = all_preds[model_names[i]] == all_labels
    b_correct = all_preds[model_names[j]] == all_labels

    n11 = ( a_correct &  b_correct).sum()
    n12 = ( a_correct & ~b_correct).sum()
    n21 = (~a_correct &  b_correct).sum()
    n22 = (~a_correct & ~b_correct).sum()

    result = mcnemar([[n11, n12], [n21, n22]], exact=False, correction=True)
    sig = result.pvalue < alpha_corrected
    pair_label = f'{model_names[i]} vs {model_names[j]}'
    pairwise_mcnemar_results[pair_label] = {
        'statistic': float(result.statistic),
        'p_value': float(result.pvalue),
        'significant_bonferroni': bool(sig),
        'table': [[int(n11), int(n12)], [int(n21), int(n22)]],
    }
    print(f'{pair_label:<35} {result.statistic:>8.2f} {result.pvalue:>10.6f} {("YES" if sig else "no"):>14}')

In [ ]:
n_bootstrap = 10_000
rng = np.random.default_rng(42)
bootstrap_accuracy_cis = {}

print(f"{'Model':<20} {'Accuracy':>10} {'95% CI':>20}")
print('-' * 55)

for name, preds in all_preds.items():
    correct = (preds == all_labels).astype(int)
    boot_accs = np.array([
        correct[rng.choice(len(correct), size=len(correct), replace=True)].mean()
        for _ in range(n_bootstrap)
    ])
    lo, hi = np.percentile(boot_accs, [2.5, 97.5])
    acc = correct.mean()
    bootstrap_accuracy_cis[name] = {
        'accuracy': float(acc),
        'ci_low': float(lo),
        'ci_high': float(hi),
    }
    print(f'{name:<20} {acc:>10.4f} [{lo:.4f}, {hi:.4f}]')

## 7. Inference Speed Benchmark

In [ ]:
model_fv = models['fastvit_t8']
if hasattr(model_fv, 'reparameterize'):
    print('Reparameterizing FastViT for inference mode...')
    model_fv.reparameterize()
elif hasattr(model_fv, 'switch_to_deploy'):
    print('Switching FastViT to deploy mode...')
    model_fv.switch_to_deploy()
else:
    print('WARNING: No reparameterize/deploy method found on FastViT.')
    print('Inference benchmark reflects TRAINING-TIME architecture.')
    print("FastViT's latency numbers will be artificially inflated.")

In [ ]:
import time

WARMUP, TIMED = 20, 100
dummy = torch.randn(1, 3, cfg['input_size'], cfg['input_size'], device=device)

for key, model in models.items():
    model.eval()
    with torch.no_grad():
        for _ in range(WARMUP):
            model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(TIMED):
            model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / TIMED * 1000
    test_results[key]['inference_ms_per_image'] = round(ms, 3)
    print(f'{key}: {ms:.2f} ms/image')

## 8. Results Summary Table

In [ ]:
import json
from sklearn.metrics import f1_score, precision_recall_fscore_support

print(f"{'Model':<18} {'Acc':>7} {'95% CI':>18} {'Macro-F1':>9} {'W-F1':>7} {'Params(M)':>10} {'ms/img':>8}")
print('-' * 90)

enriched = {}
for key, r in test_results.items():
    preds = all_preds[key]
    macro = f1_score(all_labels, preds, average='macro', zero_division=0)
    weighted = f1_score(all_labels, preds, average='weighted', zero_division=0)
    p, rec, f1_vals, sup = precision_recall_fscore_support(
        all_labels,
        preds,
        average=None,
        zero_division=0,
    )
    ci = bootstrap_accuracy_cis[key]

    print(
        f"{key:<18} {r['test_accuracy']:>7.4f} "
        f"[{ci['ci_low']:.4f}, {ci['ci_high']:.4f}] "
        f"{macro:>9.4f} {weighted:>7.4f} {r['num_params'] / 1e6:>10.1f} "
        f"{r.get('inference_ms_per_image', 'N/A'):>8}"
    )

    enriched[key] = {
        **r,
        'bootstrap_accuracy_ci': {
            'low': round(ci['ci_low'], 4),
            'high': round(ci['ci_high'], 4),
        },
        'macro_f1': round(float(macro), 4),
        'weighted_f1': round(float(weighted), 4),
        'per_class': {
            name: {
                'precision': round(float(p[i]), 4),
                'recall': round(float(rec[i]), 4),
                'f1': round(float(f1_vals[i]), 4),
                'support': int(sup[i]),
            }
            for i, name in enumerate(CLASS_NAMES)
        },
    }

enriched['_statistical_tests'] = {
    'cochran_q': {
        'statistic': round(cochran_q_result['statistic'], 6),
        'df': cochran_q_result['df'],
        'p_value': round(cochran_q_result['p_value'], 6),
    },
    'pairwise_mcnemar': {
        pair: {
            'statistic': round(result['statistic'], 6),
            'p_value': round(result['p_value'], 6),
            'significant_bonferroni': result['significant_bonferroni'],
            'table': result['table'],
        }
        for pair, result in pairwise_mcnemar_results.items()
    },
    'bootstrap_resamples': n_bootstrap,
}

metrics_path = PROJECT_ROOT / cfg['metrics_file']
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, 'w') as f:
    json.dump(enriched, f, indent=2)
print(f'\nEnriched metrics saved to {metrics_path}')

In [ ]:
# Performance vs. Efficiency scatter plot
fig, ax = plt.subplots(figsize=(7, 5))
colors = {'resnet50': '#e15759', 'tiny_vit_11m': '#4e79a7', 'fastvit_t8': '#59a14f'}
for key, r in test_results.items():
    ax.scatter(
        r['num_params'] / 1e6,
        r['test_accuracy'],
        s=r.get('inference_ms_per_image', 10) * 15,
        color=colors.get(key, 'gray'),
        alpha=0.85,
        label=key,
        zorder=3,
    )
    ax.annotate(key, (r['num_params'] / 1e6, r['test_accuracy']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xlabel('Parameters (M)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Performance vs. Efficiency (bubble size = inference time)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / cfg['figures_dir'] / 'pareto_tradeoff.png', dpi=150)
plt.show()

## 9. Grad-CAM Visualizations

Representative patches (2 per class × 4 classes) across all three models.

In [ ]:
from evaluation.gradcam import GradCAM, generate_gradcam_grid, _select_representative_patches

figures_dir = PROJECT_ROOT / cfg['figures_dir']
sample_indices = _select_representative_patches(test_ds, samples_per_class=2)

for key, model in models.items():
    print(f'Grad-CAM: {key}')
    generate_gradcam_grid(model, key, test_ds, device, figures_dir, sample_indices)

print('\nAll Grad-CAM figures saved to', figures_dir)

In [ ]:
# Display saved Grad-CAM figures inline
from IPython.display import Image as IPImage, display
for key in models:
    fig_path = figures_dir / f'gradcam_{key}.png'
    if fig_path.exists():
        print(f'\n{key}')
        display(IPImage(str(fig_path)))

## 10. Done

All outputs saved to `results/`:
- `results/metrics.json` — accuracy, per-class P/R/F1, macro/weighted F1, inference time, params
- `results/checkpoints/<model>/best_*.pt` — best checkpoints
- `results/figures/training_curves.png`
- `results/figures/confusion_matrices.png` ← NEW
- `results/figures/per_class_f1.png` ← NEW
- `results/figures/pareto_tradeoff.png`
- `results/figures/gradcam_<model>.png` (one per model)

### Statistical Tests Performed
- Cochran's Q test (omnibus: are the three error rates different?)
- Pairwise McNemar's tests with Bonferroni correction
- Bootstrap 95% confidence intervals on accuracy (10,000 resamples)

---
**LLM Disclosure:** [List here which parts of this code/notebook were generated or significantly assisted by an LLM, per course policy.]